In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# unit and sanity checks!
"""--------------------------------------------"""
# add pooled r2 -> rerun movement r2 nb
# - check that nans are still caught as different between mb/mf with out of pool averging
# - check for systematic over/under estimation of r2
# - figure out why shuffle = True leads to better results
# - note pool logic in notebook
# replicate neurotheory plots
# - clean up [r2 comp DONE], [clean up beta weight]

"""---rerun after all done & add figures to ppt---"""

# > check the fits for different regularization constants
# define responsive

# one regressor
# cvr2 with strategy model params
# add time
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id)
encoder_nopool = Encoder(subj_id, sess_id)

encoder.get_r2()
encoder.fit_encoder()
encoder_nopool.get_r2(pool=False)
encoder_nopool.fit_encoder()

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()
encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    x=encoder_nopool.scores["encoder"], y=encoder.scores["encoder"], add_unity=True
)

In [ ]:
encoder.verify()
# encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

In [ ]:
# se = ShuffledEncoder(
#     subj_id,
#     sess_id,
#     tv_keys=[
#         "response",
#         "rewarded",
#         "block_side",
#         "strategy",
#         "response_prev",
#         "rewarded_prev",
#     ],
# )
# se.plot_cvr2()
# se.plot_dr2()|
# se.plot_bound_r2()

## r2 comp between regions and strategies 

scatter version is in .verify()

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
# scores
scores = {
    k: {
        f"{reg}, {model}": encoder_.scores[model][encoder_.reg_idxs[reg]]
        for reg in encoder_.regions
        for model in ["baseline", "encoder"]
    }
    for k, encoder_ in encoders.items()
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
colors = {"DMS": "#562E9C", "DLS": "#009D51"}
styles = {
    f"{reg}, {model}": {"linestyle": linestyles[model], "color": colors[reg]}
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

for k, scores_ in scores.items():
    plot_kdes(
        scores_,
        label=rf"$r^2$, {k}",
        xlim=(-0.25, 1),
        add_means=False,
        line_kwargs=styles,
    )

In [ ]:
from core.data import colors_strategy

scores_baseline = {k: encoder_.scores["baseline"] for k, encoder_ in encoders.items()}
styles = {
    "full": {"color": "#444444", "linewidth": 1},
    "mb": {"color": colors_strategy["mb"], "linestyle": "--"},
    "mf": {"color": colors_strategy["mf"], "linestyle": "--"},
}
plot_kdes(scores_baseline, line_kwargs=styles)

## weight comp between regions and strategies

### kde

In [ ]:
# check fr of dls vs dms

In [ ]:
from core.viz import plot_kde_row
from core.data import colors_strategy
from utils.paths import FIGURES_DIR

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

# styles
colors = {"DMS": "#2383DC", "DLS": "#3DC1E2"}
styles_strategy = {f"{reg}": {"color": colors[reg]} for reg in encoder.regions}
styles_reg = {f"{k}": {"color": colors_strategy[k]} for k in encoders}

# iterate through all regressors and save
for regressor in encoder.dm_names:
    if "tents" not in regressor:
        # weights
        weights_strategy = {
            k: {
                f"{reg}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for reg in encoder_.regions
            }
            for k, encoder_ in encoders.items()
        }

        weights_reg = {
            reg: {
                f"{k}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for k, encoder_ in encoders.items()
            }
            for reg in encoder.regions
        }

        # plot
        for k, weights, styles in zip(
            ["strategy", "region"],
            [weights_strategy, weights_reg],
            [styles_strategy, styles_reg],
        ):
            fig, _ = plot_kde_row(
                weights, styles, title=rf"$\beta$ {regressor}", add_means=False
            )

            fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"
            fpath.mkdir(parents=True, exist_ok=True)
            fig.savefig(fpath / f"{regressor}_{k}.png", dpi=300, bbox_inches="tight")

            plt.close(fig)

### scatter, hist 2d, contour

In [ ]:
from core.viz import plot_scatter

# iterate through regressor and region
plot_scatter(
    weights_strategy["both"]["DLS"],
    weights_strategy["mf"]["DLS"],
    xlabel=rf"$\beta$ {regressor}, both",
    ylabel=rf"$\beta$ {regressor}, mf",
    add_unity=True,
)

In [ ]:
plt.figure()
plt.hist2d(
    weights_strategy["both"]["DLS"],
    weights_strategy["mf"]["DLS"],
    range=[[-20, 20], [-20, 20]],
    bins=50,
    cmap="Blues",
    density=True,
)
plt.colorbar()
plt.show()

In [ ]:
import seaborn as sns

x = weights_strategy["both"]["DLS"]
y = weights_strategy["mf"]["DLS"]

plt.figure()
sns.kdeplot(x=x, y=y, cmap="Blues", levels=20, thresh=0, clip=((-20, 20), (-20, 20)))
plt.show()